<a href="https://colab.research.google.com/github/scrum-exams/quant-1/blob/main/Task1_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
# 1. Load the raw dataset and check for anomalies
import pandas as pd
import numpy as np

df = pd.read_csv('input1.csv')

# Filter for a single ticker (Apple) to maintain time-series integrity
df = df[df['Name'] == 'AAPL'].copy()
df = df.sort_values('date')

# Print the number of missing values per column
print("Missing values per column:\n", df.isnull().sum())

# Print basic statistics to spot anomalous volume spikes
print("\nData description:\n", df.describe())

Missing values per column:
 date      0
open      0
high      0
low       0
close     0
volume    0
Name      0
dtype: int64

Data description:
               open         high          low        close        volume
count  1259.000000  1259.000000  1259.000000  1259.000000  1.259000e+03
mean    109.055429   109.951118   108.141589   109.066698  5.404790e+07
std      30.549220    30.686186    30.376224    30.556812  3.346835e+07
min      55.424200    57.085700    55.014200    55.789900  1.147592e+07
25%      84.647800    85.334950    84.250650    84.830650  2.969438e+07
50%     108.970000   110.030000   108.050000   109.010000  4.566893e+07
75%     127.335000   128.100000   126.290000   127.120000  6.870872e+07
max     179.370000   180.100000   178.250000   179.260000  2.668336e+08


In [5]:
# 2. Compare Forward-Fill vs Linear Interpolation on Volatility
df_ffill = df.copy().ffill()
df_interp = df.copy().interpolate(method='linear')

# Calculate 20-day rolling standard deviation of returns to compare variance
ffill_vol = df_ffill['close'].pct_change().rolling(window=20).std().var()
interp_vol = df_interp['close'].pct_change().rolling(window=20).std().var()

print(f"Variance of Volatility (Forward-Fill): {ffill_vol:.6f}")
print(f"Variance of Volatility (Linear Interp): {interp_vol:.6f}")

Variance of Volatility (Forward-Fill): 0.000022
Variance of Volatility (Linear Interp): 0.000022


/tmp/ipykernel_3367/794572684.py:3: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_interp = df.copy().interpolate(method='linear')


In [9]:
# 3. Engineering Volume-Normalized Volatility & Log Returns
# Use the interpolated dataframe moving forward
df_clean = df_interp.copy()

# Calculate daily log returns
df_clean['Log_Returns'] = np.log(df_clean['close'] / df_clean['close'].shift(1))

# Calculate volume moving average and volume-normalized volatility
df_clean['Volume_MA_20'] = df_clean['volume'].rolling(window=20).mean()
df_clean['Vol_Normalized'] = df_clean['Log_Returns'].rolling(window=20).std() / df_clean['Volume_MA_20']

# Shift features by 1 day to prevent look-ahead bias
df_clean[['Log_Returns', 'Vol_Normalized']] = df_clean[['Log_Returns', 'Vol_Normalized']].shift(1)

# Show the first few rows of the new features
print(df_clean[['date', 'close', 'Log_Returns', 'Vol_Normalized']].head(10))

            date    close  Log_Returns  Vol_Normalized
1259  2013-02-08  67.8542          NaN             NaN
1260  2013-02-11  68.5614          NaN             NaN
1261  2013-02-12  66.8428     0.010368             NaN
1262  2013-02-13  66.7156    -0.025386             NaN
1263  2013-02-14  66.6556    -0.001905             NaN
1264  2013-02-15  65.7371    -0.000900             NaN
1265  2013-02-19  65.7128    -0.013876             NaN
1266  2013-02-20  64.1214    -0.000370             NaN
1267  2013-02-21  63.7228    -0.024516             NaN
1268  2013-02-22  64.4014    -0.006236             NaN


In [10]:
# 4. Export to CSV for reconciliation check
df_clean.to_csv('engineered_output.csv', index=False)
print("File exported successfully.")

File exported successfully.
